In [17]:
import pandas as pd
import numpy as np

In [18]:
df = pd.read_excel("Online Retail.xlsx")
print("Original records:", len(df))

Original records: 541909


In [19]:
df = df.dropna(subset=["CustomerID"])
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df["TotalAmount"] = df["Quantity"] * df["UnitPrice"]

In [23]:
#Create customer-level features
customer = df.groupby("CustomerID").agg(
    Frequency = ("InvoiceNo", "nunique"),
    TotalQuantity = ("Quantity", "sum"),
    UniqueProducts = ("StockCode", "nunique"),
    Monetary = ("TotalAmount", "sum")
).reset_index()

#Average spending per order
customer["AvgBasketValue"] = (
    customer["Monetary"] / customer["Frequency"]
)

In [24]:
#Create customer segments 
customer["Segment"] = pd.qcut(
    customer["Monetary"],
    q=3,
    labels = ["Low", "Medium", "High"]
)

print("\nCustomer Segments: ")
print(customer["Segment"].value_counts())


Customer Segments: 
Segment
Low       1446
Medium    1446
High      1446
Name: count, dtype: int64


In [25]:
#Convert features into low/meium/high
features = [
    "Frequency",
    "TotalQuantity",
    "UniqueProducts",
    "Monetary",
    "AvgBasketValue"
]

for feature in features:
    customer[feature] = pd.cut(
        customer[feature],
        bins = 3,
        labels = ["Low", "Medium", "High"],
        include_lowest = True
    )

In [26]:
#Entropy function

def entropy(data):
    counts = data["Segment"].value_counts()
    total = len(data)
    ent = 0
    for count in counts:
        p = count / total
        if p > 0:
            ent -= p * np.log2(p)
    return ent

#Overall entropy
total_entropy = entropy(customer)
print("\nOverall Entropy =", round(total_entropy, 4))


Overall Entropy = 1.585


In [30]:
#Calculate information gain

results = []
for feature in features:
    weighted_entropy = 0
    for value in customer[feature].dropna().unique():
        subset = customer[customer[feature] == value]
        subset_entropy = entropy(subset)
        weight = len(subset) / len(customer)
        weight = len(subset) / len(customer)
        weighted_entropy += weight * subset_entropy
    information_gain = total_entropy - weighted_entropy
    results.append([
        feature,
        weighted_entropy,
        information_gain
    ])

In [ ]:
#Information gain table

result_table = pd.DataFrame(
    results,
    columns = [
        "Feature",
        "Weighted Entropy",
        "Information Gain"
    ]
)

result_table = result_table.sort_values(
    "Information Gain",
    ascending = False